In [1]:
import pandas as pd
import numpy as np
import os

from sklearn.metrics import (
    classification_report,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    roc_curve,
    precision_recall_curve,
    accuracy_score
)

import joblib

from sklearn.model_selection import train_test_split

from sklearn.preprocessing import StandardScaler, OneHotEncoder, MinMaxScaler

from sklearn.ensemble import (
    RandomForestClassifier,
    GradientBoostingClassifier
)

In [2]:
try:
    import shap
    print("[INFO] SHAP library found")
except ImportError:
    shap = None
    print("[INFO] SHAP library not found, skipping SHAP-related features")

print("All libraries imported successfully")

[INFO] SHAP library found
All libraries imported successfully


In [3]:
csv_file = "machine_failure_dataset.csv"

print(f"[INFO] Loading: {csv_file}")

try:
    df = pd.read_csv(csv_file)
    print(f"[INFO] Loaded shape: {df.shape}")

    # Target column
    y = df["Failure_Risk"]

    # Feature columns
    X = df.drop(
        ["Failure_Risk"],
        axis=1,
        errors="ignore"
    )

    # Encode categorical Machine_Type column
    X = pd.get_dummies(
        X,
        columns=["Machine_Type"],
        drop_first=True
    )

    FEATURES_KEY = X.columns.tolist()

    print(f"[INFO] Features used:")
    print(FEATURES_KEY)

    print("\n[INFO] Target distribution:")
    print(y.value_counts())

except FileNotFoundError:
    print(f"[ERROR] File not found: {csv_file}")

except Exception as e:
    print(f"[ERROR] {e}")

[INFO] Loading: machine_failure_dataset.csv
[INFO] Loaded shape: (1000, 6)
[INFO] Features used:
['Temperature', 'Vibration', 'Power_Usage', 'Humidity', 'Machine_Type_Lathe', 'Machine_Type_Mill']

[INFO] Target distribution:
Failure_Risk
0    700
1    300
Name: count, dtype: int64


In [4]:
X_train, X_test, y_train, y_test = train_test_split(X,y, random_state = 42, stratify = y)
print("[INFO] data split complete.")
print(f"Total Samples: {len(y)}")
print(f"Training Samples: {len(y_train)}")
print(f"Testing Samples:{len(y_test)}")

[INFO] data split complete.
Total Samples: 1000
Training Samples: 750
Testing Samples:250


In [5]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("[INFO] Data scaling complete")

# -------------------------
# Random Forest
# -------------------------
print("[INFO] Training RandomForest...")
rf = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)
rf.fit(X_train_scaled, y_train)
print("[INFO] RandomForest training complete")

[INFO] Data scaling complete
[INFO] Training RandomForest...
[INFO] RandomForest training complete


In [6]:
print("[INFO] Evaluating model on test data")
y_pred = rf.predict(X_test_scaled)
try:
    y_score = rf.predict_proba(X_test_scaled)[:,1]
except Exception:
    y_score = None
    print("[WARN] could not get predict_proba scores")
acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred, average='binary')
rec = recall_score(y_test, y_pred, average='binary')
f1 = f1_score(y_test, y_pred, average='binary')
cm = confusion_matrix(y_test, y_pred)

 # Print results
print(f"[INFO] Accuracy  : {acc:.4f}")
print(f"[INFO] Precision : {prec:.4f}")
print(f"[INFO] Recall    : {rec:.4f}")
print(f"[INFO] F1 Score  : {f1:.4f}")

print("\n[RESULT] Confusion Matrix:")
print(pd.DataFrame(cm, index = ["Actual Benign","Actual Attack"], columns = ["Pred. Benign","Pred. Attack"]))
print("\n[RESULT] Classification Report:")
print(classification_report(y_test, y_pred))

[INFO] Evaluating model on test data
[INFO] Accuracy  : 0.6600
[INFO] Precision : 0.1429
[INFO] Recall    : 0.0267
[INFO] F1 Score  : 0.0449

[RESULT] Confusion Matrix:
               Pred. Benign  Pred. Attack
Actual Benign           163            12
Actual Attack            73             2

[RESULT] Classification Report:
              precision    recall  f1-score   support

           0       0.69      0.93      0.79       175
           1       0.14      0.03      0.04        75

    accuracy                           0.66       250
   macro avg       0.42      0.48      0.42       250
weighted avg       0.53      0.66      0.57       250



In [7]:
roc_data, pr_data = {},{}
if y_score is not None:
    try:
        fpr, tpr, _ = roc_curve(y_test, y_score)
        precision_curve, recall_curve, _ = precision_recall_curve(y_test, y_score)
        roc_data = {"fpr": fpr.tolist(), "tpr": tpr.tolist()}
        pr_data = {"precision": precision_curve.tolist(),"recall": recall_curve.tolist()}
        print("[INFO] ROC and PR curve data calculated")
    except Exception as e:
        print(f"Could not compute ROC/PR data: {e}")

else:
    print("[INFO] Skipping ROC/PR calculation {no probability scores}")

[INFO] ROC and PR curve data calculated


In [8]:
top_features = []

try:
    importances = rf.feature_importances_

    # Using the FEATURES_KEY defined during data loading
    feat_imp = sorted(
        zip(FEATURES_KEY, importances),
        key=lambda x: x[1],
        reverse=True
    )[:10]

    top_features = [
        {"feature": f, "importance": float(imp)}
        for f, imp in feat_imp
    ]

    print("\n[INFO] Top important features")
    print("." * 40)

    for f, imp in feat_imp:
        print(f"{f:<30s} : {imp:.6f}")

    print("." * 40)

except Exception as e:
    print(f"[WARN] Could not extract important features: {e}")


[INFO] Top important features
........................................
Vibration                      : 0.244338
Power_Usage                    : 0.240042
Temperature                    : 0.236626
Humidity                       : 0.234099
Machine_Type_Lathe             : 0.022609
Machine_Type_Mill              : 0.022286
........................................


In [9]:
output_dir = "models"
os.makedirs(output_dir, exist_ok=True)

try:
    joblib.dump(scaler, os.path.join(output_dir, "machine_failure_scaler.joblib"))
    joblib.dump(rf, os.path.join(output_dir, "machine_failure.joblib"))
    
    if 'explainer' in locals():
        joblib.dump(explainer, os.path.join(output_dir, "shap_explainer.joblib"))

    print(f"[INFO] All models and scaler saved to '{output_dir}/' directory.")
except Exception as e:
    print(f"[ERROR] Could not save the models: {e}")
    

[INFO] All models and scaler saved to 'models/' directory.
